[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/setting_up_and_getting_data/setting_up_lsdtt3.ipynb)

# Getting lsdtt3 and lsdviztools3 set up

## First set up condacolab

This is a bit of an annoying step that should take around 2 minutes.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now we need to install `pygmt`. GMT stands for generic mapping tools. This is the slowest step in the process (will take around a minute).

In [ ]:
# Install pygmt. mamba prints hundreds of lines, so we hide them and only
# show the end of the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

Check if it works. This will give you a globe.

In [ ]:
import pygmt

fig = pygmt.Figure()
fig.basemap(region="-180/180/-90/90", projection="G10/30/15c", frame=True)
fig.coast(shorelines=True, land="grey", water="lightblue")
fig.show()

Now get lsdviztools3

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.2.0-py3-none-any.whl

In [ ]:
!pip install lsdviztools3-0.2.0-py3-none-any.whl

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.7.1.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.7.1.tar.gz

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.7.1"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's older proj.db breaks rasterio and pyproj.

In [ ]:
!lsdtt-channel-extraction -v

In [ ]:
!ls

### Reading the corners of your box

This cell defines two small functions that read latitude and longitude pairs copied from Google Maps. You do not need to edit it; just run it.

In [ ]:
import math
import re


def parse_lat_lon(text):
    """Read a "latitude, longitude" pair, as copied from Google Maps, into two numbers."""
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    if len(numbers) != 2:
        raise ValueError(
            f'Could not find a latitude and a longitude in "{text}". Right-click the place '
            'on Google Maps, click the two numbers at the top of the menu to copy them, and '
            'paste them between the quotes, for example "55.9476, -3.1582".')
    latitude = float(numbers[0])
    longitude = float(numbers[1])
    if not -90 <= latitude <= 90:
        raise ValueError(f'"{text}": the first number is the latitude and must be between '
                         '-90 and 90. Google Maps gives the latitude first.')
    if not -180 <= longitude <= 180:
        raise ValueError(f'"{text}": the second number is the longitude and must be '
                         'between -180 and 180.')
    return latitude, longitude


def box_from_corners(lower_left, upper_right):
    """Return west, east, south and north for the box with these two opposite corners."""
    lat_1, lon_1 = parse_lat_lon(lower_left)
    lat_2, lon_2 = parse_lat_lon(upper_right)
    if lat_1 == lat_2 or lon_1 == lon_2:
        raise ValueError("The two corners must have different latitudes and different longitudes.")
    # min and max, so you get the same box whichever pair of opposite corners you pick
    west, east = min(lon_1, lon_2), max(lon_1, lon_2)
    south, north = min(lat_1, lat_2), max(lat_1, lat_2)
    width_km = (east - west) * 111.32 * math.cos(math.radians((south + north) / 2))
    height_km = (north - south) * 110.57
    print(f"Box: west {west:.5f}, east {east:.5f}, south {south:.5f}, north {north:.5f}")
    print(f"Size: about {width_km:.0f} km east-west by {height_km:.0f} km north-south")
    return west, east, south, north

### Edit code block below for your site!

**Choosing the box with two right-clicks on Google Maps:**

1. On Google Maps, right-click the **lower left (south-west)** corner of your area. The first line of the menu that appears is the latitude and longitude, for example `55.9476273103734, -3.158220317286066`. Click it to copy it.
2. Paste it between the quotes after `lower_left =` (Ctrl-V, or Cmd-V on a Mac).
3. Do the same for the **upper right (north-east)** corner, after `upper_right =`.

If you pick the other two corners instead (upper left and lower right), you get the same box.

In [ ]:
# Right-click the lower left (south-west) corner on Google Maps, copy, paste:
lower_left = "36.55, -3.56"
# ... and the upper right (north-east) corner:
upper_right = "37.45, -2.44"

west, east, south, north = box_from_corners(lower_left, upper_right)


In [ ]:
# Create a parameter file for lsdtt-fetch-raster
with open("sierra_nevada_params.txt", "w") as f:
    f.write("write_fname: sierra_nevada\n")   # output is sierra_nevada_DEM.tif
    f.write(f"west: {west}\n")
    f.write(f"east: {east}\n")
    f.write(f"south: {south}\n")
    f.write(f"north: {north}\n")
    f.write("dem_source: cop30_aws\n")       # Copernicus GLO-30, no API key needed
    f.write("target_epsg: 32630\n")          # UTM zone 30N
    f.write("grid_spacing: 30\n")

# Run lsdtt-fetch-raster. The first argument is the directory holding the parameter file.
!lsdtt-fetch-raster ./ sierra_nevada_params.txt

**A note on the sea.** The Copernicus DEM does not mark the sea as nodata: it stores it as an elevation of exactly **0 m**. This box reaches the Mediterranean, so about a fifth of `sierra_nevada_DEM.tif` is sea at 0 m. Before computing slopes, channels or statistics, set those pixels to nodata, or the sea will be treated as a huge flat surface. `lsdtt-raster-preprocessing` does this with `min_elevation: 0`, writing `sierra_nevada_remove_seas.tif`:

```
read_fname: sierra_nevada_DEM.tif
write_fname: sierra_nevada
min_elevation: 0
```